# Phase 1.7: Pickle-First Bootstrap Null Calibration for v2a-RSN

This notebook calibrates the observed graph-instability statistic without rerunning the Phase 0 connectivity notebooks.

Workflow:

1. Load each completed `{recording}.pkl` adjacency grid and compute the observed statistic directly from it.
2. Load the filtered raw traces only to fit a VAR(1) moving-block residual null.
3. Re-estimate connectivity for each surrogate using the exact method parameters recorded in `run_metadata.json`.
4. Save every surrogate adjacency grid as an independent checkpoint, allowing interrupted runs to resume.
5. Export global critical values, pointwise depth bands, summary tables, and dynamic figures for all recording-method combinations.

The notebook does not read `transitions.csv`; conditioning depths come from the keys of the completed pickle dictionaries.

In [1]:
from __future__ import annotations

import json
import logging
import platform
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
)
logger = logging.getLogger(__name__)

PROJECT_ROOT = Path('.').resolve()
while not (PROJECT_ROOT / 'src' / 'markovianity_diagnostic').exists():
    parent = PROJECT_ROOT.parent
    if parent == PROJECT_ROOT:
        raise FileNotFoundError('Could not find the hidden-confounding-diagnostics project root')
    PROJECT_ROOT = parent

sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from markovianity_diagnostic.experiments.v2a_calibration import (  # noqa: E402
    calibration_payload,
    discover_complete_recordings,
    load_v2a_calibration_input,
    make_v2a_surrogate_analyzer,
    plot_v2a_calibration_grid,
    plot_v2a_pointwise_grid,
    run_resumable_v2a_calibration,
    stable_seed,
    write_calibration_payload,
)

print(f'Project root: {PROJECT_ROOT}')

Project root: /Users/sadiqadedayo/Documents/projects/Detecting Latent confounders with Markovianity/hidden-confounding-diagnostics


In [2]:
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'calibration' / 'v2a'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

METHOD_DIRS = ['c-GC', 'c-GC-star']
P_VALUES = [1, 2, 3, 4, 5, 6, 7]
P0 = 1
B = 1
BLOCK_LENGTH = 50
SEED = 42
N_JOBS = 1

# Set to a subset such as ['220119_F2_run11'] for a timed smoke run.
# Completed checkpoints remain reusable when B is increased later.
RECORDINGS = None

print(f'Output directory: {OUTPUT_DIR}')
print(f'Methods: {METHOD_DIRS}')
print(f'Depths: {P_VALUES}; p0={P0}')
print(f'Bootstrap: B={B}, block_length={BLOCK_LENGTH}, n_jobs={N_JOBS}')

Output directory: /Users/sadiqadedayo/Documents/projects/Detecting Latent confounders with Markovianity/hidden-confounding-diagnostics/outputs/calibration/v2a
Methods: ['c-GC', 'c-GC-star']
Depths: [1, 2, 3, 4, 5, 6, 7]; p0=1
Bootstrap: B=1, block_length=50, n_jobs=1


In [3]:
available_recordings = discover_complete_recordings(
    PROJECT_ROOT,
    method_dirs=METHOD_DIRS,
    p_values=P_VALUES,
)
recordings = available_recordings if RECORDINGS is None else list(RECORDINGS)
unknown = sorted(set(recordings).difference(available_recordings))
if unknown:
    raise ValueError(f'Requested recordings do not have complete pickle grids: {unknown}')

print(f'Complete recordings ({len(recordings)}):')
for recording in recordings:
    print(f'  - {recording}')

Complete recordings (4):
  - 220119_F2_run11
  - 220127_F4_run2
  - 220210_F1_run6
  - 220210_F2_run5


In [ ]:
results = {method_dir: {} for method_dir in METHOD_DIRS}
summary_rows = []
run_output_paths = []
suite_start = time.perf_counter()

for method_index, method_dir in enumerate(METHOD_DIRS, start=1):
    print(f'\n[{method_index}/{len(METHOD_DIRS)}] {method_dir}')
    for recording_index, recording in enumerate(recordings, start=1):
        print(f'  [{recording_index}/{len(recordings)}] {recording}')
        run_start = time.perf_counter()

        calibration_input = load_v2a_calibration_input(
            PROJECT_ROOT,
            recording=recording,
            method_dir=method_dir,
            p_values=P_VALUES,
        )
        analyze_surrogate = make_v2a_surrogate_analyzer(calibration_input.metadata)
        run_seed = stable_seed(SEED, method_dir, recording)
        checkpoint_dir = OUTPUT_DIR / 'checkpoints' / method_dir / recording
        config_metadata = {
            'recording': recording,
            'method_dir': method_dir,
            'gcstar_params': calibration_input.metadata['gcstar_params'],
            'inferred_completion': calibration_input.metadata.get('inferred_completion'),
        }

        outcome = run_resumable_v2a_calibration(
            calibration_input.X,
            observed_adjacencies=calibration_input.observed_adjacencies,
            analyze_surrogate=analyze_surrogate,
            p_values=P_VALUES,
            p0=P0,
            B=B,
            block_length=BLOCK_LENGTH,
            seed=run_seed,
            checkpoint_dir=checkpoint_dir,
            n_jobs=N_JOBS,
            config_metadata=config_metadata,
        )

        elapsed = time.perf_counter() - run_start
        run_metadata = {
            **config_metadata,
            'connectivity_pickle': str(calibration_input.connectivity_path.relative_to(PROJECT_ROOT)),
            'trace_shape_time_by_neurons': list(calibration_input.X.shape),
            'p_values': P_VALUES,
            'p0': P0,
            'B': B,
            'block_length': BLOCK_LENGTH,
            'seed': run_seed,
            'elapsed_seconds': elapsed,
        }
        payload = calibration_payload(outcome, metadata=run_metadata)
        run_output_path = OUTPUT_DIR / method_dir / recording / 'bootstrap.json'
        write_calibration_payload(run_output_path, outcome, metadata=run_metadata)
        run_output_paths.append(run_output_path)
        results[method_dir][recording] = payload

        summary_rows.append({
            'recording': recording,
            'method': method_dir,
            'T_obs': outcome.result.observed['T_obs'],
            'critical_90': outcome.result.null['critical_90'],
            'critical_95': outcome.result.null['critical_95'],
            'critical_99': outcome.result.null['critical_99'],
            'p_value': outcome.result.null['p_value'],
            'reject_global_95': outcome.result.diagnosis['reject_global_95'],
            'first_exceedance_depth': outcome.result.diagnosis['first_exceedance_depth'],
            'B': B,
            'reused_replicates': outcome.reused_replicates,
            'elapsed_seconds': elapsed,
        })
        print(
            f"    T_obs={outcome.result.observed['T_obs']:.6f}; "
            f"critical_95={outcome.result.null['critical_95']:.6f}; "
            f"p={outcome.result.null['p_value']:.4f}; "
            f"reused={outcome.reused_replicates}/{B}; elapsed={elapsed:.1f}s"
        )

suite_elapsed = time.perf_counter() - suite_start
print(f'\nCalibration suite completed in {suite_elapsed:.1f}s')


[1/2] c-GC
  [1/4] 220119_F2_run11


2026-07-04 10:28:37,994 - markovianity_diagnostic.experiments.v2a_rsn_utils - INFO - Loaded traces from 220119_F2_F2_run11_cells_fluorescence_signals.npy: shape (1010, 3613)
2026-07-04 10:28:38,002 - markovianity_diagnostic.experiments.v2a_rsn_utils - INFO - Selecting 100 identified neurons from 1010 total
2026-07-04 10:28:38,012 - markovianity_diagnostic.experiments.v2a_rsn_utils - INFO - Dropping 12 bad frames
2026-07-04 10:28:38,162 - markovianity_diagnostic.experiments.v2a_calibration - INFO - Running bootstrap replicate 1/1 (seed=2693083017)
2026-07-04 10:28:38,247 - markovianity_diagnostic.experiments.v2a_calibration - INFO - Inferring surrogate depth 1/7 (P=1)


In [ ]:
bootstrap_results_path = OUTPUT_DIR / 'bootstrap_results.json'
bootstrap_results_tmp = OUTPUT_DIR / '.bootstrap_results.json.tmp'
bootstrap_results_tmp.write_text(
    json.dumps(
        {
            'schema_version': 2,
            'analysis': 'pickle-first v2a moving-block bootstrap calibration',
            'methods': results,
        },
        indent=2,
        sort_keys=True,
    ),
    encoding='utf-8',
)
bootstrap_results_tmp.replace(bootstrap_results_path)

summary_df = pd.DataFrame(summary_rows).sort_values(['recording', 'method'])
summary_path = OUTPUT_DIR / 'bootstrap_summary.csv'
summary_df.to_csv(summary_path, index=False)

histogram_path = plot_v2a_calibration_grid(
    results,
    OUTPUT_DIR / 'bootstrap_T_obs.png',
)
depth_bands_path = plot_v2a_pointwise_grid(
    results,
    OUTPUT_DIR / 'depth_bands.png',
)

print(summary_df.to_string(index=False))
print(f'Wrote {bootstrap_results_path}')
print(f'Wrote {summary_path}')
print(f'Wrote {histogram_path}')
print(f'Wrote {depth_bands_path}')

In [ ]:
input_paths = []
for method_dir in METHOD_DIRS:
    for recording in recordings:
        input_paths.append(
            str((PROJECT_ROOT / 'outputs' / 'v2a-RSNs' / method_dir / f'{recording}.pkl').relative_to(PROJECT_ROOT))
        )
for recording in recordings:
    input_paths.extend(
        str(path.relative_to(PROJECT_ROOT))
        for path in sorted((PROJECT_ROOT / 'data' / 'v2a-RSNs' / recording).glob('*cells_fluorescence_signals.npy'))
    )

is_full_production_run = set(recordings) == set(available_recordings) and B >= 50
manifest = {
    'created_at': datetime.now(timezone.utc).isoformat(),
    'analysis': 'bootstrap_null_v2a_pickle_first',
    'status': 'complete' if is_full_production_run else 'smoke_or_subset',
    'input_contract': 'observed adjacency pickles plus raw traces for null fitting',
    'observed_connectivity_recomputed': False,
    'methods': METHOD_DIRS,
    'recordings': recordings,
    'method_params': {
        'B': B,
        'p_values': P_VALUES,
        'p0': P0,
        'null_model': 'moving-block residual bootstrap',
        'block_length': BLOCK_LENGTH,
        'n_jobs': N_JOBS,
    },
    'random_seed': SEED,
    'input_paths': input_paths,
    'output_paths': [
        str(bootstrap_results_path.relative_to(PROJECT_ROOT)),
        str(summary_path.relative_to(PROJECT_ROOT)),
        str(histogram_path.relative_to(PROJECT_ROOT)),
        str(depth_bands_path.relative_to(PROJECT_ROOT)),
        *(str(path.relative_to(PROJECT_ROOT)) for path in run_output_paths),
    ],
    'software_versions': {
        'python': platform.python_version(),
        'numpy': np.__version__,
        'pandas': pd.__version__,
    },
    'elapsed_seconds': suite_elapsed,
}
manifest_path = OUTPUT_DIR / 'manifest.json'
manifest_tmp = OUTPUT_DIR / '.manifest.json.tmp'
manifest_tmp.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
manifest_tmp.replace(manifest_path)
print(f'Wrote {manifest_path}')

In [ ]:
expected_outputs = [
    bootstrap_results_path,
    summary_path,
    histogram_path,
    depth_bands_path,
    manifest_path,
    *run_output_paths,
]
missing_outputs = [path for path in expected_outputs if not path.exists()]
if missing_outputs:
    raise FileNotFoundError(f'Missing calibration outputs: {missing_outputs}')

expected_rows = len(METHOD_DIRS) * len(recordings)
assert len(summary_df) == expected_rows, (
    f'Expected {expected_rows} recording-method rows, found {len(summary_df)}'
)
assert summary_df['B'].eq(B).all()
assert summary_df['p_value'].between(0, 1).all()

for method_dir in METHOD_DIRS:
    for recording in recordings:
        checkpoint_dir = OUTPUT_DIR / 'checkpoints' / method_dir / recording
        checkpoint_count = len(list(checkpoint_dir.glob('replicate_*.pkl')))
        assert checkpoint_count >= B, (
            f'{method_dir}/{recording} has {checkpoint_count} checkpoints; expected at least {B}'
        )

print(f'Verified {len(expected_outputs)} output files and {expected_rows} calibration rows.')